# Step 3: Line and leader characteristics

This step collects the mass, stiffness and diameter profiles of the fly
line and the leader, and assembles them with a rod into a *rig* that the
simulation can use.

**Limits of this step today.** One fly line (a WF5 profile, scalable to
other AFTM weights) and one leader are bundled. There is no tool yet for
measuring your own line; you can write your own component files in the
format shown below.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

from flycastsim.fem import (
    bundled_rig_path,
    copy_example_rig,
    load_component,
    load_rig,
)

## Components are data files

A component is a JSON file with metadata plus a CSV table with the
profile along the component. A `rig.json` lists the components and how
they are joined.

In [ ]:
rig_path = bundled_rig_path("cast1")
print(rig_path.read_text())

In [ ]:
print((rig_path.parent / "line.json").read_text())

## Load the line and the leader

`load_component` returns the profile as arrays: arc length `s` [m], mass
per length `m` [kg/m], bending stiffness `EI` [N m²] and diameter `d` [m].

In [ ]:
line = load_component(rig_path.parent / "line.json")
leader = load_component(rig_path.parent / "leader.json")
for comp in (line, leader):
    total_g = np.trapezoid(comp.m, comp.s) * 1000
    print(f"{comp.kind:7s} {comp.length:5.2f} m, {total_g:5.2f} g: {comp.name}")

## Scale the line to an AFTM weight

The line's mass profile is scaled so that the first 30 ft match the AFTM
standard for the chosen line weight. A heavier line loads the rod more.

In [ ]:
fig = go.Figure()
for weight in (4, 5, 6, 7):
    scaled = load_component(rig_path.parent / "line.json", aftm_weight=weight)
    fig.add_trace(go.Scatter(x=scaled.s, y=scaled.m * 1000,
                             mode="lines+markers", name=f"AFTM {weight}"))
fig.update_layout(title="Fly line mass profile",
                  xaxis_title="distance from rod tip [m]",
                  yaxis_title="mass per length [g/m]")
fig

In [ ]:
fig = go.Figure()
for comp in (line, leader):
    fig.add_trace(go.Scatter(x=comp.s, y=comp.d * 1000,
                             mode="lines+markers", name=comp.kind))
fig.update_layout(title="Diameter profiles (used for air drag)",
                  xaxis_title="distance along the component [m]",
                  yaxis_title="diameter [mm]")
fig

## Assemble a rig

`load_rig` joins rod, line and leader into one multi-domain beam. The
rod and line are joined by a pinned hinge, the line and leader are welded.

In [ ]:
rig = load_rig("cast1", aftm_weight=6)
print(rig.name)
print(f"{rig.n_subdomains} subdomains, {rig.n_nodes} nodes")
for sub, junction in zip(rig.subdomains, [*rig.junctions, None]):
    print(f"  {sub.length:5.2f} m, {sub.n_nodes} nodes"
          + (f"  -> {junction.kind} joint" if junction else ""))

## Make your own rig

`copy_example_rig` copies the bundled rig to a folder you can edit. Change
the CSV tables or the JSON metadata, then pass the path of the new
`rig.json` to `load_rig` or to the cast simulation (`rig=...`).

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    my_rig = copy_example_rig(Path(tmp) / "my_rig")
    print(sorted(p.name for p in my_rig.parent.iterdir()))
    print(load_rig(my_rig).n_nodes, "nodes in the copied rig")